<a href="https://colab.research.google.com/github/natiihegab-gif/Matricula---Gasto-educativo/blob/main/Copia_de_Matriculas_2011_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gasto público en educación por matriculado en términos reales (2011-2024)

## Objetivo

Construir un indicador anual de gasto público educativo real por alumno para cada provincia argentina, combinando información de matrícula educativa y gasto público destinado a educación.

El objetivo es analizar la evolución del financiamiento educativo provincial considerando los cambios en la cantidad de estudiantes y descontando el efecto de la inflación.

---

## Fuente de datos

### Matrícula educativa

Bases anuales de matrícula del sistema educativo argentino (2011-2024).

Información utilizada:

- Provincia
- Año
- Cantidad de matriculados

### Gasto educativo

Datos de gasto público provincial destinado a educación.

Información utilizada:

- Provincia
- Año
- Gasto educativo en pesos corrientes

### Índice de precios

Índice utilizado para convertir el gasto educativo nominal a términos reales.

Información utilizada:

- Año
- Índice de precios
- Año base seleccionado para la deflactación

---

## Metodología

### 1. Construcción de la matrícula educativa provincial

- Lectura automatizada de los archivos anuales de matrícula.
- Identificación dinámica de variables asociadas a la cantidad de alumnos.
- Cálculo de matrícula total por establecimiento educativo.
- Agregación de la matrícula a nivel provincial.
- Construcción del total nacional.

### 2. Cálculo del gasto educativo real

- Integración de la información de gasto educativo provincial.
- Conversión del gasto nominal a precios constantes mediante un índice de precios.
- Expresión del gasto educativo en términos reales utilizando un año base determinado.

### 3. Cálculo del gasto real por matriculado

Se construye el siguiente indicador:

$$Gasto\ real\ por\ matriculado_{it} =\frac{Gasto\ educativo\ real_{it}}{Matrícula_{it}}$$

Donde:

- $i$: provincia.
- $t$: año.

El indicador permite analizar cuánto gasto educativo real corresponde, en promedio, a cada estudiante matriculado en cada jurisdicción.

---

## Tecnologías utilizadas

- Python
- Pandas
- Google Colab
- GitHub

---

## Resultado

Base final con información anual por provincia:

| Variable | Descripción |
|----------|-------------|
| Provincia | Jurisdicción educativa |
| Año | Año del relevamiento |
| Matrícula | Cantidad total de estudiantes matriculados |
| Gasto educativo nominal | Gasto destinado a educación en pesos corrientes |
| Gasto educativo real | Gasto educativo ajustado por inflación |
| Gasto real por matriculado | Gasto educativo real promedio por estudiante |

---

## Análisis posibles

La base permite estudiar:

- Evolución del gasto educativo real por estudiante.
- Diferencias de financiamiento entre provincias.
- Brechas regionales en inversión educativa.
- Relación entre crecimiento del gasto y evolución de la matrícula.
- Cambios en la inversión educativa provincial en el período 2011-2024.

## Librerias

In [24]:
import pandas as pd
from pathlib import Path
from google.colab import drive

## Matriculas

In [25]:
# ===========================
# 1. Montar Drive
# ===========================
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
from pathlib import Path
import pandas as pd

# ============================================
# Carpeta donde están los archivos
# ============================================

carpeta = Path("/content/drive/MyDrive/EDUCACION/MATRICULA")

# ============================================
# 1. Procesar archivo histórico (2003-2010)
# ============================================

historico = pd.read_excel(
    carpeta / "/content/drive/MyDrive/EDUCACION/MATRICULA/cantidad de alumnos matriculados.xlsx",
    header=None
)

# La primera fila contiene los nombres de las columnas
historico.columns = ["anio"] + historico.iloc[0, 1:].tolist()

# Eliminar la fila de encabezados
historico = historico.iloc[1:].reset_index(drop=True)

# Pasar de formato ancho a largo
historico = historico.melt(
    id_vars="anio",
    var_name="provincia",
    value_name="matriculados"
)

# Limpiar nombres
historico["provincia"] = (
    historico["provincia"]
        .astype(str)
        .str.strip()
        .replace({
            "Total País": "Total Nacional"
        })
)

historico["anio"] = pd.to_numeric(historico["anio"], errors="coerce")
historico["matriculados"] = pd.to_numeric(
    historico["matriculados"],
    errors="coerce"
)

historico = historico.dropna(subset=["anio"])

historico["anio"] = historico["anio"].astype(int)

# ============================================
# 2. Procesar archivos 2011-2024
# ============================================

resultados = []

for archivo in sorted(carpeta.glob("*.xlsx")):

    # Saltar el archivo histórico
    if archivo.name == "cantidad de alumnos matriculados.xlsx":
        continue

    anio = int(archivo.stem[:4])

    print(f"\nProcesando {archivo.name}...")

    df = pd.read_excel(archivo)

    # Normalizar nombres de columnas
    df.columns = (
        df.columns
          .astype(str)
          .str.strip()
          .str.lower()
    )

    excluir = ["provincia", "departamento", "sector", "ambito"]
    prefijos = ("v_", "r_", "s_", "_sec", "multi")

    matriculas = [
        c for c in df.columns
        if c not in excluir
        and not c.startswith(prefijos)
    ]

    print(f"{anio}: {len(matriculas)} variables de matrícula")

    # Convertir a numérico
    df[matriculas] = (
        df[matriculas]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
    )

    # Matrícula total
    df["matricula_total"] = df[matriculas].sum(axis=1)

    # Total por provincia
    provincia = (
        df.groupby("provincia", as_index=False)["matricula_total"]
          .sum()
          .rename(columns={"matricula_total": "matriculados"})
    )

    provincia["anio"] = anio

    # Agregar total nacional
    total = pd.DataFrame({
        "provincia": ["Total Nacional"],
        "matriculados": [provincia["matriculados"].sum()],
        "anio": [anio]
    })

    provincia = pd.concat(
        [provincia, total],
        ignore_index=True
    )

    resultados.append(provincia)

# ============================================
# 3. Unir todo
# ============================================

matricula_2011_2024 = pd.concat(
    resultados,
    ignore_index=True
)

matricula_total = pd.concat(
    [historico, matricula_2011_2024],
    ignore_index=True
)

# ============================================
# 4. Ordenar
# ============================================

matricula_total = (
    matricula_total
    .sort_values(["anio", "provincia"])
    .reset_index(drop=True)
)

print(matricula_total.head())
print(matricula_total.tail())


Procesando 2011 Matricula - agregada.xlsx...
2011: 20 variables de matrícula

Procesando 2012 Matricula - agregada.xlsx...
2012: 20 variables de matrícula

Procesando 2013 Matricula - agregada.xlsx...
2013: 20 variables de matrícula

Procesando 2014 Matricula - agregada.xlsx...
2014: 20 variables de matrícula

Procesando 2015 Matricula - agregada.xlsx...
2015: 20 variables de matrícula

Procesando 2016 Matricula - agregada.xlsx...
2016: 20 variables de matrícula

Procesando 2017 Matricula - agregada.xlsx...
2017: 20 variables de matrícula

Procesando 2018 Matricula - agregada.xlsx...
2018: 20 variables de matrícula

Procesando 2019 Matricula - agregada.xlsx...
2019: 20 variables de matrícula

Procesando 2020 Matricula - agregada.xlsx...
2020: 20 variables de matrícula

Procesando 2021 Matricula - agregada.xlsx...
2021: 20 variables de matrícula

Procesando 2022 Matricula - agregada.xlsx...
2022: 20 variables de matrícula

Procesando 2023 Matricula - agregada.xlsx...
2023: 21 variables

## Gasto en Educación Real